# Bearing fault diagnosis — fast-kurtogram demodulation band (Colab)

Self-contained: no project files needed. A CPU runtime is enough.

For every 1-second window at the main operating condition (1500 rpm, 0.7 Nm, 1000 N), selects the demodulation
band with the fast kurtogram (Antoni, 2007), which searches over band centres *and* widths, and computes the four
fault-frequency features in that band (suffix `_fk`).

It uses the windows cached in Drive by `colab_cnn.ipynb` (`data/raw_windows_N15_M07_F10/`); bearings not cached
yet are downloaded first (~160 MB each). Results are saved per bearing, so the notebook can be re-run after a
disconnection.

At the end, download `data/features/kurtogram_N15_M07_F10.csv` from Drive into the project's `data/features/`
folder and run `python scripts/add_kurtogram_features.py`.

Data licence: CC BY-NC 4.0 — cite Lessmeier et al. (2016) and the Paderborn KAt-DataCenter.

## 1. Setup

In [ ]:
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null

import shutil
from pathlib import Path

import numpy as np
import pandas as pd

DRIVE_DATA = Path("/content/drive/MyDrive/load_bearing_research_paper/data")
CACHE_DIR = DRIVE_DATA / "raw_windows_N15_M07_F10"       # cached 1 s windows, one .npz per bearing
OUT_DIR = DRIVE_DATA / "features" / "kurtogram_parts"    # one file per bearing
OUT_CSV = DRIVE_DATA / "features" / "kurtogram_N15_M07_F10.csv"
FS = 64_000
SHAFT_HZ = 1500 / 60

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Bearings (same as the project's splits)

In [ ]:
"""Bearings, bearing-level splits and calibration rotations for the CNN runs.

A self-contained copy of the project's split definitions (bearing_uq.bearings and bearing_uq.splits);
tests/test_cnn_protocol.py checks that both give the same splits and rotations.
"""

from itertools import product

import numpy as np
import pandas as pd

HEALTHY, INNER, OUTER = "healthy", "inner_race", "outer_race"
LABELS = [HEALTHY, INNER, OUTER]  # class index = position in this list

BEARINGS = pd.DataFrame(
    [(c, HEALTHY, "none") for c in ["K001", "K002", "K003", "K004", "K005", "K006"]]
    + [(c, OUTER, "artificial") for c in ["KA01", "KA03", "KA05", "KA06", "KA07", "KA08", "KA09"]]
    + [(c, INNER, "artificial") for c in ["KI01", "KI03", "KI05", "KI07", "KI08"]]
    + [(c, OUTER, "real") for c in ["KA04", "KA15", "KA16", "KA22", "KA30"]]
    + [(c, INNER, "real") for c in ["KI04", "KI14", "KI16", "KI17", "KI18", "KI21"]],
    columns=["bearing", "label", "origin"],
)

SOURCE_HEALTHY = ["K001", "K002", "K004"]
TARGET_HEALTHY = ["K003", "K005", "K006"]


def _codes(label, origin):
    rows = BEARINGS[(BEARINGS.label == label) & (BEARINGS.origin == origin)]
    return sorted(rows.bearing)


ARTIFICIAL_INNER = _codes(INNER, "artificial")
ARTIFICIAL_OUTER = _codes(OUTER, "artificial")
SOURCE = SOURCE_HEALTHY + ARTIFICIAL_OUTER + ARTIFICIAL_INNER
TARGET = TARGET_HEALTHY + _codes(OUTER, "real") + _codes(INNER, "real")


def rotations():
    """Yield (key, calibration bearings, training bearings) for all 3 x 5 x 7 = 105 rotations.

    The key ("K001+KI01+KA01") matches the file names used by the project's other experiments.
    """
    for calib in product(SOURCE_HEALTHY, ARTIFICIAL_INNER, ARTIFICIAL_OUTER):
        train = sorted(set(SOURCE) - set(calib))
        yield "+".join(calib), list(calib), train


def random_window_split(meta, test_fraction=0.2, seed=0):
    """Sanity check only: a random split of windows in which every bearing appears in both training and test.

    For each bearing (sorted by code), its windows (sorted by recording and window) are shuffled and the first
    test_fraction are marked as test. Returns a boolean array aligned with meta (True = test). Depends only on the
    window keys, so the project gives the same split for the feature table as Colab does for the raw windows.
    """
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(meta), dtype=bool)
    for code in sorted(meta["bearing"].unique()):
        rows = meta.index[meta["bearing"] == code]
        rows = meta.loc[rows].sort_values(["recording", "window"]).index.to_numpy()
        n_test = int(round(test_fraction * len(rows)))
        is_test[meta.index.get_indexer(rng.permutation(rows)[:n_test])] = True
    return is_test

## 4. Loading and caching windows (same as colab_cnn.ipynb)

In [ ]:
"""Download Paderborn recordings and cache their 1-second vibration windows (one .npz per bearing).

Windows are cut exactly as in the feature table: recordings 1-20, non-overlapping 1 s windows from the
start of each recording, unreadable or missing recordings skipped.
"""

import shutil
import subprocess
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

FS = 64_000
WINDOW = FS                 # 1-second windows
RECORDINGS = 20
CONDITION = "N15_M07_F10"   # main operating condition: 1500 rpm, 0.7 Nm, 1000 N
BASE_URL = "https://groups.uni-paderborn.de/kat/BearingDataCenter/"


def _channel_name(ch):
    name = ch["Name"]
    while isinstance(name, np.ndarray):
        name = name.flat[0]
    return str(name)


def load_vibration(path, channel="vibration_1"):
    record = loadmat(path)[Path(path).stem][0, 0]
    for ch in record["Y"].ravel():
        if _channel_name(ch) == channel:
            return np.asarray(ch["Data"], dtype=float).ravel()
    raise KeyError(f"{channel} not in {[_channel_name(c) for c in record['Y'].ravel()]}")


def windows_from_folder(folder, code, condition=CONDITION):
    """All 1 s windows of one bearing: (x float32 (n, WINDOW), recording numbers, window numbers)."""
    xs, recs, wins = [], [], []
    for n in range(1, RECORDINGS + 1):
        hits = list(Path(folder).rglob(f"{condition}_{code}_{n}.mat"))
        if not hits:
            print(f"  missing {condition}_{code}_{n}.mat")
            continue
        try:
            signal = load_vibration(hits[0])
        except Exception as e:  # a few Paderborn files are unreadable
            print(f"  skipped {hits[0].name}: {e}")
            continue
        for w in range(len(signal) // WINDOW):
            xs.append(signal[w * WINDOW:(w + 1) * WINDOW].astype(np.float32))
            recs.append(n)
            wins.append(w)
    return np.stack(xs), np.array(recs), np.array(wins)


def cache_bearing(code, cache_dir, condition=CONDITION):
    """Download one bearing's archive, extract it, and save its windows to <cache_dir>/<code>.npz."""
    with tempfile.TemporaryDirectory() as tmp:
        archive = Path(tmp) / f"{code}.rar"
        with urllib.request.urlopen(BASE_URL + f"{code}.rar") as r, open(archive, "wb") as f:
            shutil.copyfileobj(r, f, length=1 << 20)
        out = Path(tmp) / "x"
        out.mkdir()
        subprocess.run(["unrar", "x", "-o+", "-inul", str(archive), f"{out}/"], check=True)
        archive.unlink()
        x, rec, win = windows_from_folder(out, code, condition)
    np.savez(Path(cache_dir) / f"{code}.npz", x=x, recording=rec, window=win)
    return len(x)


def load_cache(cache_dir, bearings):
    """Stack the cached windows of all bearings: (x (N, WINDOW) float32, meta DataFrame)."""
    xs, metas = [], []
    for b in bearings.itertuples():
        d = np.load(Path(cache_dir) / f"{b.bearing}.npz")
        xs.append(d["x"])
        metas.append(pd.DataFrame({"bearing": b.bearing, "label": b.label, "origin": b.origin,
                                   "recording": d["recording"], "window": d["window"]}))
    return np.concatenate(xs), pd.concat(metas, ignore_index=True)

## 5. Bearing geometry and fault frequencies

In [ ]:
"""Bearing geometry and characteristic fault frequencies."""

import math
from dataclasses import dataclass


@dataclass(frozen=True)
class BearingGeometry:
    n_balls: int
    ball_diameter_mm: float
    pitch_diameter_mm: float
    contact_angle_deg: float = 0.0

    @property
    def _ratio(self) -> float:
        return (self.ball_diameter_mm / self.pitch_diameter_mm) * math.cos(
            math.radians(self.contact_angle_deg)
        )


# Deep groove ball bearing 6203 used in the Paderborn test rig.
BEARING_6203 = BearingGeometry(n_balls=8, ball_diameter_mm=6.75, pitch_diameter_mm=28.55)


def bpfo(shaft_hz: float, geom: BearingGeometry = BEARING_6203) -> float:
    """Ball pass frequency, outer race (Hz)."""
    return geom.n_balls / 2 * (1 - geom._ratio) * shaft_hz


def bpfi(shaft_hz: float, geom: BearingGeometry = BEARING_6203) -> float:
    """Ball pass frequency, inner race (Hz)."""
    return geom.n_balls / 2 * (1 + geom._ratio) * shaft_hz


def bsf(shaft_hz: float, geom: BearingGeometry = BEARING_6203) -> float:
    """Ball spin frequency (Hz)."""
    d, D = geom.ball_diameter_mm, geom.pitch_diameter_mm
    return D / (2 * d) * (1 - geom._ratio**2) * shaft_hz


def ftf(shaft_hz: float, geom: BearingGeometry = BEARING_6203) -> float:
    """Fundamental train (cage) frequency (Hz)."""
    return 0.5 * (1 - geom._ratio) * shaft_hz

## 6. Envelope spectrum

In [ ]:
"""Envelope spectrum: band-pass, Hilbert envelope, then FFT."""

import numpy as np
from scipy.signal import butter, hilbert, sosfiltfilt

# Default demodulation band (Hz). A fixed band keeps the pipeline simple;
# a kurtogram-selected band is a possible later refinement.
DEFAULT_BAND = (2_000.0, 10_000.0)


def envelope_spectrum(x: np.ndarray, fs: float,
                      band: tuple[float, float] = DEFAULT_BAND
                      ) -> tuple[np.ndarray, np.ndarray]:
    """Return (frequencies, amplitudes) of the envelope spectrum."""
    low, high = band
    sos = butter(4, [low, high], btype="bandpass", fs=fs, output="sos")
    filtered = sosfiltfilt(sos, np.asarray(x, dtype=float))
    env = np.abs(hilbert(filtered))
    env -= env.mean()
    amps = np.abs(np.fft.rfft(env)) / len(env)
    freqs = np.fft.rfftfreq(len(env), d=1 / fs)
    return freqs, amps

## 7. Fault-frequency features

In [ ]:
"""Energy at characteristic fault frequencies in the envelope spectrum.

Each value is the peak amplitude near k * f_fault, divided by the median
amplitude of the envelope spectrum over a reference range. Values well above
1 indicate a periodic impact at that fault frequency.
"""

import numpy as np

REFERENCE_RANGE = (10.0, 500.0)  # Hz, for the noise-floor median
TOLERANCE = 0.02                 # +/- 2 % search window around each target


def peak_near(freqs: np.ndarray, amps: np.ndarray, target: float,
              tolerance: float = TOLERANCE) -> float:
    lo, hi = target * (1 - tolerance), target * (1 + tolerance)
    mask = (freqs >= lo) & (freqs <= hi)
    return float(amps[mask].max()) if mask.any() else 0.0


def noise_floor(freqs: np.ndarray, amps: np.ndarray,
                ref: tuple[float, float] = REFERENCE_RANGE) -> float:
    mask = (freqs >= ref[0]) & (freqs <= ref[1])
    floor = float(np.median(amps[mask])) if mask.any() else 0.0
    return floor if floor > 0 else 1e-12


def fault_band_features(freqs: np.ndarray, amps: np.ndarray,
                        fault_freqs: dict[str, float],
                        harmonics: int = 2) -> dict[str, float]:
    """e.g. fault_freqs={"bpfo": 76.4, "bpfi": 123.6} -> bpfo_h1, bpfo_h2, ..."""
    floor = noise_floor(freqs, amps)
    out = {}
    for name, f in fault_freqs.items():
        for k in range(1, harmonics + 1):
            out[f"{name}_h{k}"] = peak_near(freqs, amps, k * f) / floor
    return out

## 8. Fast kurtogram

In [ ]:
"""Demodulation band from the fast kurtogram (Antoni, 2007): searches over band centres *and* widths.

The fast kurtogram evaluates the kurtosis of the complex envelope in every band of a 1/3-binary tree. At level k
the range 0..fs/2 is split into 2^k bands; at the intermediate level k + log2(3) - 1 (written k.6) into 3 * 2^(k-1)
bands. Levels: 0, 1, 1.6, 2, 2.6, 3, ... The band with the largest kurtosis is where impacts stand out most from
the background. Antoni computes the bands with a tree of short quasi-analytic filters; here each band's complex
envelope is taken directly from the FFT (ideal band-pass, decimated), which gives the same band grid with sharper
band edges.

As for the spectral-kurtosis band (spectral_kurtosis.py), frequencies outside F_MIN..F_MAX are excluded, and
bands narrower than MIN_WIDTH are skipped so that the envelope keeps the second harmonic of the inner-race
frequency (about 250 Hz at 1500 rpm) on both sides of the carrier.
"""

import numpy as np


F_MIN, F_MAX = 1_000.0, 30_000.0
MIN_WIDTH = 1_000.0  # Hz


def band_grid(fs: float, min_width: float = MIN_WIDTH) -> list[tuple[float, float, float]]:
    """(level, low, high) for every band of the 1/3-binary tree that is at least min_width wide."""
    nyq = fs / 2
    grid = [(0.0, 0.0, nyq)]
    k = 1
    while nyq / 2 ** k >= min_width or nyq / (3 * 2 ** (k - 1)) >= min_width:
        for n_bands, level in ((2 ** k, float(k)), (3 * 2 ** (k - 1), k + np.log2(3) - 1)):
            width = nyq / n_bands
            if width >= min_width:
                grid += [(level, i * width, (i + 1) * width) for i in range(n_bands)]
        k += 1
    return grid


def kurtogram(x: np.ndarray, fs: float, f_min: float = F_MIN, f_max: float = F_MAX,
              min_width: float = MIN_WIDTH) -> list[tuple[float, float, float, float]]:
    """(level, low, high, kurtosis) per band, each band clipped to f_min..f_max.

    Kurtosis of the complex envelope c: E|c|^4 / (E|c|^2)^2 - 2, which is 0 for Gaussian noise.
    """
    x = np.asarray(x, dtype=float)
    spectrum = np.fft.rfft(x - x.mean())
    freqs = np.fft.rfftfreq(len(x), d=1 / fs)
    out = []
    for level, low, high in band_grid(fs, min_width):
        low, high = max(low, f_min), min(high, f_max)
        if high - low < min_width:
            continue
        band = spectrum[(freqs >= low) & (freqs < high)]
        power = np.abs(np.fft.ifft(band)) ** 2
        mean_power = power.mean()
        k = (power ** 2).mean() / mean_power ** 2 - 2 if mean_power > 0 else 0.0
        out.append((level, low, high, float(k)))
    return out


def kurtogram_band(x: np.ndarray, fs: float) -> tuple[float, float, float]:
    """(low, high, level) of the band with the largest kurtosis."""
    level, low, high, _ = max(kurtogram(x, fs), key=lambda row: row[3])
    return low, high, level


def kurtogram_features(x: np.ndarray, fs: float, fault_freqs: dict[str, float]) -> dict[str, float]:
    """Fault features (suffix _fk) from the kurtogram-selected band, plus the band's edges and level."""
    low, high, level = kurtogram_band(x, fs)
    freqs, amps = envelope_spectrum(x, fs, (low, high))
    out = {f"{k}_fk": v for k, v in fault_band_features(freqs, amps, fault_freqs).items()}
    out.update(fk_band_low_hz=low, fk_band_high_hz=high, fk_level=level)
    return out

## 9. Cache any windows not cached yet (skipped if colab_cnn.ipynb already did it)

In [ ]:
for b in BEARINGS.itertuples():
    if (CACHE_DIR / f"{b.bearing}.npz").exists():
        print(f"{b.bearing}: already cached")
        continue
    try:
        print(f"{b.bearing}: {cache_bearing(b.bearing, CACHE_DIR)} windows")
    except Exception as e:
        print(f"{b.bearing}: FAILED - {e}  (run this cell again)")

## 10. Kurtogram band and features for every window (a few minutes)

In [ ]:
FAULT_FREQS = {"bpfo": bpfo(SHAFT_HZ), "bpfi": bpfi(SHAFT_HZ)}
print({k: round(v, 2) for k, v in FAULT_FREQS.items()})

for b in BEARINGS.itertuples():
    part = OUT_DIR / f"{b.bearing}.csv"
    if part.exists():
        continue
    d = np.load(CACHE_DIR / f"{b.bearing}.npz")
    rows = [{"bearing": b.bearing, "recording": int(r), "window": int(w),
             **kurtogram_features(x.astype(float), FS, FAULT_FREQS)}
            for x, r, w in zip(d["x"], d["recording"], d["window"])]
    pd.DataFrame(rows).to_csv(part, index=False)
    print(f"{b.bearing}: {len(rows)} windows")

## 11. Combine into one table

In [ ]:
table = pd.concat([pd.read_csv(OUT_DIR / f"{c}.csv") for c in BEARINGS.bearing], ignore_index=True)
table.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV} ({len(table)} windows)")

## 12. Sanity check

Median features per group, and the bands the kurtogram chose.

In [ ]:
t = table.merge(BEARINGS, on="bearing")
print(t.groupby(["origin", "label"])[["bpfo_h1_fk", "bpfi_h1_fk"]].median().round(2))
print()
print("selected band width (Hz):")
print((t.fk_band_high_hz - t.fk_band_low_hz).describe().round(0))
print()
print("selected band centre (kHz), median per bearing:")
print(((t.fk_band_low_hz + t.fk_band_high_hz) / 2000).groupby(t.bearing).median().round(1).to_string())